In [36]:
import json
import datetime
from typing import TypedDict, List, Dict, Any, Optional, Annotated
import operator
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from server.database.connection import get_db_conn
from psycopg2.extras import RealDictCursor          # excute() 결과를 dictionary 형태로 받기 위함

# LLM 모델을 초기화합니다.
llm = ChatOpenAI(model="gpt-4.1-mini")

# State 정의
-> example structure는 추후에 노드에서 structured output으로 넣도록 하면 됨.

In [37]:
class AnlysisState(TypedDict):
    """혜택 분석 워크플로우 상태 정의"""
    
    # --- 입력 (Input) ---
    user_id: str
    store_name: str # location_name과 동일
    store_category: str # raw_category와 동일 (카카오 API 원본 문자열)

    # --- 중간 데이터 (Intermediate Data) ---
    # 1. DB에서 조회한 사용자의 카드 후보 목록 (search_user_cards 노드 결과)
    candidate_cards: List[Dict[str, Any]]

    # 2. 현장 결제 이벤트 (crawl_offline_events 노드 결과)
    offline_events: Annotated[List[Dict[str, Any]], operator.add]

    # 3. 카드별 분석 결과 (consolidate_analysis 노드 결과)
    analyzed_cards: List[Dict[str, Any]]

    # --- 최종 출력 (Final Output) ---
    # 4. LLM이 생성한 최종 순위 및 분석 결과 (generate_final_ranking 노드 결과)
    final_ranking: Dict[str, Any]

    # 5. 사용자에게 보여줄 최종 브리핑 문자열 (format_briefing 노드 결과)
    final_briefing: str

In [38]:
class CardAnalysisResult(BaseModel):
    """하나의 카드에 대한 모든 분석 결과를 통합하는 모델 (consolidate_analysis 노드의 출력)"""
    card_id: int
    card_name: str
    card_type: str = Field(description="카드 타입 ('credit' 또는 'check')")
    is_benefit_applicable: bool = Field(description="적용 가능한 혜택이 있는지")
    is_limit_remaining: bool = Field(description="월간/연간 혜택 한도 잔여 여부 (유효한 혜택이 하나라도 남아있는지 여부)")
    is_rules_verified_by_rag: bool = Field(description="RAG를 통해 약관상 예외 조항이 없음을 확인했는지 여부")
    rag_validation_details: str = Field(description="RAG 검증 시 발견된 주의사항 또는 확인 내용")
    applicable_benefits: List[Dict[str, Any]] = Field(description="해당 가맹점에서 적용 가능하며 한도도 남아있는 혜택 목록과 상세 조건")
    final_eligibility: bool = Field(description="모든 조건을 종합했을 때 최종 혜택 적용 가능 여부")

class Recommendation(BaseModel):
    """LLM이 생성하는 개별 추천 항목 모델"""
    rank: int = Field(description="혜택 순위")
    payment_method: str = Field(description="추천하는 결제 수단 이름 (예: 'KB 토심이 카드' 또는 '네이버페이 현장결제')")
    benefit_description: str = Field(description="예상 혜택에 대한 설명. 금액이 특정되지 않을 경우, 할인율이나 조건을 명시. 예: '10% 할인', '2만원 이상 결제 시 20% 할인이 2000원 할인보다 유리'")
    positive_reason: str = Field(description="이 결제 수단을 추천하는 긍정적인 이유")
    critical_review: str = Field(description="놓칠 수 있는 단점이나 주의사항 (비판적 전략). 예를 들어, '이 혜택을 받으면 실적에서 제외됩니다.' 또는 '다른 혜택과 중복 적용되지 않습니다.' 등. 단점이 없다면 '특별한 단점 없음'으로 명시.")
    evidence: str = Field(description="이 추천의 근거가 된 데이터나 분석 내용 요약")

class FinalRanking(BaseModel):
    """LLM의 최종 판단 결과를 구조화하는 최상위 모델 (generate_final_ranking 노드의 출력)"""
    recommendations: List[Recommendation] = Field(description="순위화된 추천 목록 (최대 3개)")
    summary: str = Field(description="사용자를 위한 최종 요약 및 조언")

## Node 정의

In [39]:
def search_user_cards(state: AnlysisState) -> List[Dict[str, Any]]:
    """DB에서 사용자의 카드 정보 조회"""
    print("--- 카드 정보 조회 중 ---")
    user_id = state["user_id"]
    try:
        with get_db_conn() as conn:
            cursor = conn.cursor(cursor_factory=RealDictCursor)     # 결과를 딕셔너리 형태로 받기 위해(tuple -> dictionary)
            cursor.execute("""
                SELECT u.id AS user_card_id, c.card_id AS card_id, c.card_name, u.last_month_spent, u.issue_date, c.card_type, c.benefits_json
                    FROM user_card u LEFT JOIN card_master c 
                           ON u.card_id = c.id
                WHERE u.user_id = %s""", (user_id,))
            
            candidate_cards = [dict(row) for row in cursor.fetchall()]   # RealDictCursor[]~ -> 일반 딕셔너리 리스트로 변환
            
            print(f"✅ 조회된 카드 수: {len(candidate_cards)}개")
            return {"candidate_cards" : candidate_cards}
    except Exception as e:
        print(f"❌ DB에서 사용자 카드 데이터 조회 실패: {e}")
        raise e

In [ ]:
# 가장 처음 파라미터로 넘겨줄 입력값을 정의합니다.
state = {
    "user_id": "1", # 테스트용 사용자 ID (실제 DB에 있는 user_id)
    "store_name": "세븐일레븐 강남점",
    "store_category": "CS2", # 카페
    "candidate_cards": [],
    "offline_events": [],
    "analyzed_cards": [],
    "final_ranking": {},
    "final_briefing": ""
}

candidate_cards = search_user_cards(state)
state["candidate_cards"] = candidate_cards
print(state)

In [40]:
def valid_benefit(card: Dict[str, Any], store_category: str) -> List[Dict[str, Any]]:
    """카드 한 개에 대해, 주어진 결제 정보에 적용 가능한 모든 혜택을 찾아 요약"""
    benefits_json = card.get("benefits_json", {})
    if not benefits_json:
        return []

    card_name = card.get("card_name")
    last_month_spent = card.get("last_month_spent", 0)

    category_map = {
        'FD6': "FOOD", 'CE7': "DAILY_LIFE_GROUP", 'CS2': "DAILY_LIFE_GROUP",
        'HP8': "MEDICAL", 'PM9': "MEDICAL", 'MT1': "DAILY_LIFE_GROUP",
        'AC5': "EDUCATION", 'PK6': "PARKING_LOT", 'OL7': "OIL", 
        'SW8': "TRANSPORTATION", 'CT1': "DAILY_LIFE_GROUP"
    }
    target_category = category_map.get(store_category, "OTHER")  # 사용자가 선택한 업종

    print(f"💳 {card_name} 혜택 분석")
    found_benefits = []         # 해당 업종과 실적 조건을 충족하는 혜택들을 모두 찾아서 요약

    for benefit in benefits_json.get("benefits", []):
        if benefit.get("category") != target_category:
            continue

        conditions = benefit.get("conditions", {})
        min_performance = conditions.get("min_performance", 0)      # 실적 조건
        
        if last_month_spent < min_performance:
            continue                # 해당 업종이 다른 혜택 안에도 있을 수 있어서
        
        benefit_summary = {}
        benefit_type = benefit.get("type")          # PERCENT_DISCOUNT, FIXED_DISCOUNT, FREE_ACCESS, FEE_WAIVER 등
        benefit_value = benefit.get("value")        # 할인되는 금액, 횟수, 퍼센트
        unit = benefit.get("unit")                  # WON, PERCENT, COUNT 등
        
        desc = "혜택 확인 불가"     # default
        if benefit_type == "PERCENT_DISCOUNT":
            desc = f"{benefit_value}% 할인"
        elif benefit_type == "KRW_DISCOUNT":
            desc = f"{benefit_value}원 할인"
        elif benefit_type == "CASHBACK":
            desc = f"{benefit_value}{'%' if unit == 'PERCENT' else '원'} 캐시백"
        elif benefit_type == "POINT_ACCUMULATION":
            desc = f"{benefit_value}{'%' if unit == 'PERCENT' else '점'} 적립"
        elif benefit_type == "FREE_ACCESS":
            desc = f"무료 이용 {benefit_value}회"
        elif benefit_type == "FEE_WAIVER":
            desc = f"수수료 {benefit_value}% 면제"

        if conditions.get("per_transaction_cap"):       # 건당 최대 할인 금액
            desc += f" (건당 최대 {conditions['per_transaction_cap']:,}원)"

        benefit_summary['description'] = desc
        benefit_summary['benefit_id'] = benefit.get('benefit_id')       # 혜택 제공명(ex. 수수료 우대)
        
        merchants = benefit.get("merchant", [])
        if merchants:
            benefit_summary['applicable_merchants'] = ", ".join(merchants)
        
        limits = benefit.get("limits", {})      # 1구간은 얼마, 2구간은 얼마
        limit_desc_parts = []           # 구간 별 혜택 한도
        LIMIT_TYPES = ["monthly_performance_tiers", "monthly", "yearly"]

        for key in LIMIT_TYPES:
            value = limits.get(key)
            # 값이 없거나 None이면 다음 키로 넘어감
            if value is None:
                continue
            
            if key == "monthly_performance_tiers":
                # 리스트 형태인 '실적별 한도' 처리
                tiers = [f"{t['tier_min']//10000}만원↑ 월 {t['limit']:,}원" for t in value]
                limit_desc_parts.append(f"실적별: {', '.join(tiers)}")
                
            elif key == "monthly":
                # 숫자 형태인 '월간 한도' 처리
                limit_desc_parts.append(f"월간: {value:,}원")
                
            elif key == "yearly":
                # 숫자 형태인 '연간 한도' 처리
                limit_desc_parts.append(f"연간: {value:,}원")

            # 4. 최종 결과 합치기
            if limit_desc_parts:
                benefit_summary['limit_info'] = " / ".join(limit_desc_parts)
            else:
                benefit_summary['limit_info'] = "한도 제한 없음"

        performance_impact = benefit.get("performance_impact", {})      # 결제건이 실적에 미치는 영향
        perf_comment = performance_impact.get("comment")
        if not perf_comment:
            counts = performance_impact.get("counts_toward_performance", True)
            all_or_nothing = performance_impact.get("is_all_or_nothing_exclusion", False)

            if not counts:
                if all_or_nothing:
                    perf_comment = "할인 적용 시 결제 건 전체 실적 제외"
                else:
                    perf_comment = "할인 적용된 금액만 실적 제외"
            else:
                perf_comment = "실적에 포함됨"
        benefit_summary['performance_impact'] = perf_comment
        print(f"✨{card_name}의 {target_category} 혜택 정리: {benefit_summary}")
        found_benefits.append(benefit_summary)

    return found_benefits

In [ ]:
def check_benefit_limits(user_card_id: int, user_card_name:str, applicable_benefits: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """DB에서 월별/연별 한도 사용량을 조회하여 실제 남은 한도 확인"""
    if not applicable_benefits:
        print(f"\n😢 {user_card_name} 카드로는 적용될 혜택이 없습니다.")
        return [] # 적용될 혜택이 없으면 빈 리스트 반환

    # 현재 월을 'YYYY-MM' 형식으로 가져옵니다.
    import datetime
    current_month = datetime.datetime.now().strftime('%Y-%m')

    benefit_ids = [b['benefit_id'] for b in applicable_benefits if 'benefit_id' in b]
    if not benefit_ids:
        print("\n😢 혜택 id가 없어서 조회가 불가능합니다.")
        return applicable_benefits # 혜택 ID가 없으면 한도 제한 없이 통과

    try:
        with get_db_conn() as conn:
            cursor = conn.cursor(cursor_factory=RealDictCursor)
            # 여러 benefit_id에 대해 한 번에 조회
            cursor.execute("""
                SELECT benefit_id, remaining_limit, remaining_count
                FROM monthly_benefit_usage
                WHERE user_card_id = %s
                  AND base_month = %s
                  AND benefit_id = ANY(%s)
            """, (user_card_id, current_month, benefit_ids))
            
            # RealDictRow 객체를 완벽한 dict 타입으로 캐스팅하며 benefit_id를 key로 매핑
            usage_records = {rec['benefit_id']: dict(rec) for rec in cursor.fetchall()}
            valid_benefits = []
            
            for benefit in applicable_benefits:
                benefit_id = benefit.get('benefit_id')
                if benefit_id in usage_records:           # 사용자가 혜택을 사용하지 않았으면 해당 테이블에 값이 없음
                    record = usage_records[benefit_id]
                    remaining_limit = record.get('remaining_limit')
                    remaining_count = record.get('remaining_count')

                    # 1. 한도 소진 여부 판단
                    is_limit_exhausted = (
                        (remaining_limit is not None and remaining_limit <= 0) or
                        (remaining_count is not None and remaining_count <= 0)
                    )

                    if is_limit_exhausted:
                        print(f"⚠️ {user_card_id}의 {benefit_id} 혜택 한도 초과")
                        continue  # 한도가 소진된 혜택은 스킵

                    # 2. LLM 분석용 데이터 주입
                    if remaining_limit is not None:
                        benefit['current_remaining_limit'] = remaining_limit

                    if remaining_count is not None:
                        benefit['current_remaining_count'] = remaining_count
                else:
                    benefit['current_remaining_info'] = "이번 달 사용 이력 없음 (최대 한도 보유)"
                
                valid_benefits.append(benefit)
            
            print(f"\n✅ 한도 확인 완료: {user_card_name} (유효한 혜택 {len(valid_benefits)}개)")
            print("-----------------------------------------")
            return valid_benefits

    except Exception as e:
        print(f"❌ DB에서 월별 혜택 한도 조회 실패: {e}")
        # 예외 발생 시 안전하게 한도가 없다고 가정하여 False 반환
        return []

In [ ]:
for card in state["candidate_cards"]:
    print(f"--- 카드 분석 시작: {card['card_name']} ---")
    
    # 1. 먼저, 현재 카드에 적용 가능한 혜택만 가져옵니다.
    applicable_benefits = valid_benefit(card, state["store_category"])

    # 2. 그 다음, 해당 카드의 혜택 목록을 check_benefit_limits에 전달하여 한도를 확인합니다.
    valid_benefits_with_limit_check = check_benefit_limits(
        user_card_id=card["user_card_id"],
        user_card_name=card["card_name"],
        applicable_benefits=applicable_benefits
    )

In [42]:
def cross_check_with_rag(card: Dict[str, Any], store_name: str, store_category: str) -> Dict[str, Any]:
    """pgvector를 이용해 혜택과 주의사항을 RAG로 교차 검증"""
    benefits_json = card.get("benefits_json", {})
    critical_warning = benefits_json.get("critical_warning", "특별한 주의사항 없음.")
    
    # 1. DB 필터링용 영문 카테고리 코드 (예: "FOOD")
    filter_category_map = {
        'FD6': "FOOD", 'CE7': "DAILY_LIFE_GROUP", 'CS2': "DAILY_LIFE_GROUP",
        'HP8': "MEDICAL", 'PM9': "MEDICAL", 'MT1': "DAILY_LIFE_GROUP",
        'AC5': "EDUCATION", 'PK6': "PARKING_LOT", 'OL7': "OIL", 
        'SW8': "TRANSPORTATION", 'CT1': "DAILY_LIFE_GROUP"
    }
    
    # 자연어 임베딩용 한국어 카테고리 (예: "음식점")
    embedding_category_map = {
        'FD6': "음식점", 'CE7': "카페", 'CS2': "편의점",
        'HP8': "병원", 'PM9': "약국", 'MT1': "대형마트",
        'AC5': "학원", 'PK6': "주차장", 'OL7': "주유소/충전소", 
        'SW8': "교통", 'CT1': "문화"
    }
    
    filter_category = filter_category_map.get(store_category, 'OTHER')
    embedding_category = embedding_category_map.get(store_category, '일반매장')
    
    # 1. 임베딩 모델 초기화 및 쿼리 벡터 생성
    try:
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        
        query_text = f"카드명: {card.get('card_name')} | 혜택: {embedding_category} | 상세: {store_name} 혜택 조건, 실적 제외 대상 및 주의사항"
        query_vector = embeddings.embed_query(query_text)
        
    except Exception as e:
        print(f"❌ 임베딩 생성 실패: {e}")
        # 임베딩 실패 시 기본 정보만 반환
        return {
            "is_rules_verified_by_rag": False,
            "rag_validation_details": f"기본 주의사항: {critical_warning} (RAG 검증 실패: {e})"
        }
    
    print(f"\n🕵️ '{store_name}'에 대한 '{card.get('card_name')}' RAG 검증 진행")
    vector_search_result = ""
    try:
        with get_db_conn() as conn:
            cursor = conn.cursor(cursor_factory=RealDictCursor)
            
            # LIKE 검색을 위해 영문 카테고리 앞뒤로 % 추가
            like_pattern = f"%{filter_category}%"
            
            # 2. pgvector를 사용한 벡터 유사도 검색 실행
            cursor.execute("""
                SELECT content, 1 - (embedding <=> %s::vector) as similarity
                FROM card_benefit_vectors
                WHERE card_id = %s 
                    AND benefit_category LIKE %s
                ORDER BY embedding <=> %s::vector
                LIMIT 1
            """, (str(query_vector), str(card.get('card_id')), like_pattern, str(query_vector)))
            
            row = cursor.fetchone()
            
            # 유사도가 특정 임계값(예: 0.75) 이상일 때만 유의미한 정보로 간주
            if row and row['similarity'] > 0.75:
                vector_search_result = dict(row).get('content', '')
                print(f"🔍 RAG 검증: '{store_name}'에 대한 약관 유사도 {row['similarity']:.2f}의 내용 발견")
            else:
                print(f"🔍 RAG 검증: '{store_name}'에 대한 연관성 높은 약관 내용 없음")

    except Exception as e:
        print(f"❌ RAG 벡터 검색 실패: {e}")

    final_details = f"기본 주의사항: {critical_warning}"
    if vector_search_result:
        # 사용자가 이해하기 쉽게 포맷팅
        final_details += f" | ⚡ 추가 확인사항: {vector_search_result}"

    return {
        "is_rules_verified_by_rag": True, # RAG 검증 시도는 항상 했으므로 True, 내용은 details에 포함
        "rag_validation_details": final_details
    }

In [ ]:
for card in state["candidate_cards"]:
    valid_benefits_with_limit_check = cross_check_with_rag(
        card=card,
        store_name=state["store_name"],
        store_category=state["store_category"]
    )

In [43]:
def consolidate_analysis(state: AnlysisState) -> dict:
    """카드별 규칙(실적,한도), RAG 검증을 수행하고 분석 결과 통합"""
    print("\n--- 분석 결과 통합 및 RAG 검증 중 ---")

    analyzed_cards = []
    for card in state["candidate_cards"]:
        # 1. 적용 가능한 혜택 찾기
        applicable_benefits = valid_benefit(card, state["store_category"])
        
        # 2. RAG를 통한 교차 검증
        rag_check = cross_check_with_rag(card, state["store_name"], state["store_category"])
        
        # 3. DB에서 실시간 한도 조회
        valid_applicable_benefits = check_benefit_limits(card['user_card_id'], card['card_name'], applicable_benefits)

        # 4. 최종 분석 결과 종합
        is_benefit_applicable_overall = len(applicable_benefits) > 0        
        is_limit_remaining_overall = len(valid_applicable_benefits) > 0
        final_eligibility = is_benefit_applicable_overall and is_limit_remaining_overall and rag_check['is_rules_verified_by_rag']

        result = CardAnalysisResult(
            card_id=card['card_id'],
            card_name=card['card_name'],
            card_type=card.get('card_type', 'credit'),
            is_benefit_applicable=is_benefit_applicable_overall,
            is_limit_remaining=is_limit_remaining_overall,
            is_rules_verified_by_rag=rag_check['is_rules_verified_by_rag'],
            rag_validation_details=rag_check['rag_validation_details'],
            applicable_benefits=valid_applicable_benefits,
            final_eligibility=final_eligibility
        )
        analyzed_cards.append(result.model_dump())      # Pydantic 모델을 딕셔너리로 변환하여 저장
    
    print(f"=> ✅ 분석 결과 통합 완료: {len(analyzed_cards)}개 카드 분석됨")
    return {"analyzed_cards": analyzed_cards}

In [ ]:
analyzed_card = consolidate_analysis(state)
state["analyzed_cards"] = analyzed_card

print(state)

In [44]:
def generate_final_ranking(state: AnlysisState) -> dict:
    """통합된 모든 정보를 바탕으로 LLM이 비판적 전략을 사용하여 최종 순위를 생성합니다."""
    print("\n--- LLM 최종 순위 생성 중 ---")
    
    structured_llm = llm.with_structured_output(FinalRanking)
    
    prompt = f"""
    당신은 최고의 카드 혜택 분석 전문가입니다. 아래 정보를 바탕으로 사용자에게 가장 유리한 결제 수단을 최대 3개까지 순위별로 추천해주세요. 
    결제 금액이 정해져 있지 않으므로, 할인율과 고정 할인 금액을 비교하여 조건부 추천을 할 수 있습니다. (예: '2만원 이상 결제 시 A카드가 유리, 미만 시 B카드가 유리')
    
    **규칙:**
    1. 각 추천 항목에 대해 '비판적 검토(critical_review)'를 반드시 포함해야 합니다. 예상되는 단점이나 주의사항(예: 실적 제외, 다른 혜택과 중복 불가 등)을 명확히 지적해주세요. 단점이 없다면 '특별한 단점 없음'으로 표기하세요.
    2. 최종 요약(summary)을 통해 왜 이 순위가 최선인지 사용자에게 친절하게 설명해주세요.
    3. `benefit_description` 필드에는 구체적인 할인율이나 금액 대신, 사용자에게 가장 와닿는 형태의 혜택 설명을 요약해서 제공해주세요.
    4. 입력 데이터에 매장명에 지점명(ex. 강남점)이 포함된 경우, 다음 순서로 처리한다(카드 혜택, 현장 결제 이벤트 모두 동일하게 적용):
        - 먼저 카드 혜택(및 현장 결제 이벤트)에 해당 지점명이 명시된 조건이 있는지 확인한다.
        - 지점명 관련 조건이 존재하면, 해당 조건을 기준으로 혜택을 판단한다.
        - 지점명 관련 조건이 없다면, 매장명에서 지점명을 제거하고 “순수 매장명” 기준으로 다시 혜택을 확인한다.
    5. 현장 결제 이벤트 관련 정보가 없다면 일단 넘어가세요.
    6. 혜택 적용 카테고리로 받은 값은 아래의 내용을 보고 그에 해당 하는 키워드로 바꿔 판단에 참고하세요.
        - 'FD6': FOOD (음식점)
        - 'CE7', 'CS2', 'MT1', 'CT1': DAILY_LIFE_GROUP (카페, 편의점, 대형마트, 문화)
        - 'HP8', 'PM9': MEDICAL (병원, 약국)
        - 'AC5': EDUCATION (학원)
        - 'PK6': PARKING_LOT (주차장)
        - 'OL7': OIL (주유소/충전소)
        - 'SW8': TRANSPORTATION (교통)

    **입력 정보:**
    - 사용자 ID: {state['user_id']}
    - 결제 장소: {state['store_name']}
    - 혜택 적용 카테고리: {state['store_category']} 
    - 분석된 카드 혜택: {json.dumps(state['analyzed_cards'], indent=2, ensure_ascii=False)}
    - 현장 결제 이벤트: {json.dumps(state.get('offline_events', []), indent=2, ensure_ascii=False)} 
    """

    response = structured_llm.invoke(prompt) 

    print(f"✅ LLM이 추천하는 혜택 순위 생성 완료")
    return {"final_ranking": response.model_dump()}

In [ ]:
final_ranking = generate_final_ranking(state)
print(final_ranking)
state["final_ranking"] = final_ranking
print(state)

In [45]:
def format_briefing(state: AnlysisState) -> dict:
    """ LLM이 생성한 구조화된 순위 결과를 사용자가 보기 좋은 문자열로 변환합니다."""
    print("\n--- 최종 브리핑 포맷팅 중 ---")
    ranking_data = state["final_ranking"]
    briefing = f"✨ **'{state['store_name']}' 최적 결제 플랜** ✨\n\n"
    
    for rec in ranking_data['recommendations']:
        briefing += f"**🏆 {rec['rank']}순위: {rec['payment_method']}**\n"
        briefing += f"- 혜택 내용: **{rec['benefit_description']}**\n"
        briefing += f"- 추천 이유: {rec['positive_reason']}\n"
        briefing += f"- ⚠️ **체크포인트**: {rec['critical_review']}\n"
        briefing += f"- 근거: {rec['evidence']}\n\n"
    
    briefing += f"**💡 최종 요약**\n{ranking_data['summary']}"
    
    print(briefing)
    return {"final_briefing": briefing}

In [ ]:
final_briefing = format_briefing(state)
state["final_briefing"] = final_briefing
print(state)

## Graph 생성 및 실행

In [46]:
def create_benefit_analysis_graph():
    workflow = StateGraph(AnlysisState)

    # Add nodes
    # 데이터 수집 단계 (병렬 실행)
    # workflow.add_node("gather_data", RunnableParallel(
    #     candidate_cards=search_user_cards,
    #     offline_events=fetch_offline_events_from_rag
    # ))

    workflow.add_node(search_user_cards)
    # 분석 및 통합 단계
    workflow.add_node(consolidate_analysis)
    # LLM 추천 및 순위 생성 단계
    workflow.add_node(generate_final_ranking)
    # 최종 결과 포맷팅 단계
    workflow.add_node(format_briefing)

    # Set entry point
    # workflow.set_entry_point("gather_data")
    workflow.set_entry_point("search_user_cards")

    # Add edges
    workflow.add_edge("search_user_cards", "consolidate_analysis")
    # workflow.add_edge("gather_data", "consolidate_analysis")
    workflow.add_edge("consolidate_analysis", "generate_final_ranking")
    workflow.add_edge("generate_final_ranking", "format_briefing")
    workflow.add_edge("format_briefing", END)

    return workflow.compile()

app = create_benefit_analysis_graph()

initial_input = {
    "user_id": "1",
    "store_name": "홈플러스 남현점",
    "store_category": "MT1"
}

print("\n🚀 워크플로우 실행 시작...")
final_state = app.invoke(initial_input)


🚀 워크플로우 실행 시작...
--- 카드 정보 조회 중 ---
✅ 조회된 카드 수: 2개

--- 분석 결과 통합 및 RAG 검증 중 ---
💳 K-패스카드(체크) 혜택 분석
✨K-패스카드(체크)의 DAILY_LIFE_GROUP 혜택 정리: {'description': '5.0% 캐시백 (건당 최대 1,000원)', 'benefit_id': 'TELECOM_FEES_5_CASHBACK', 'applicable_merchants': 'SKT, KT, LG U+', 'limit_info': '월간: 3,000원', 'performance_impact': '통신요금 캐시백은 전월실적 산정 대상 포함.'}
✨K-패스카드(체크)의 DAILY_LIFE_GROUP 혜택 정리: {'description': '5.0% 캐시백 (건당 최대 10,000원)', 'benefit_id': 'CONVENIENCE_STORE_5_CASHBACK', 'applicable_merchants': 'GS25, CU', 'limit_info': '월간: 1,000원', 'performance_impact': '편의점 캐시백은 전월실적 산정 대상 포함.'}

🕵️ '홈플러스 남현점'에 대한 'K-패스카드(체크)' RAG 검증 진행
🔍 RAG 검증: '홈플러스 남현점'에 대한 연관성 높은 약관 내용 없음

✅ 한도 확인 완료: K-패스카드(체크) (유효한 혜택 2개)

-----------------------------------------
💳 KB Youth Club 체크카드 B팩 혜택 분석
✨KB Youth Club 체크카드 B팩의 DAILY_LIFE_GROUP 혜택 정리: {'description': '20% 할인', 'benefit_id': 'CONVENIENCE_STORE', 'applicable_merchants': 'GS25, CU', 'limit_info': '월간: 2,000원', 'performance_impact': 'KB Pay 오프라인 매장 결제 시 제공; 일부 매장 제